In [1]:
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import json
import os
import csv
import shutil
import re
import requests
from datetime import datetime
from textwrap import wrap
from sentence_transformers import SentenceTransformer
from paddleocr import PaddleOCR
from sklearn.metrics.pairwise import cosine_similarity
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# === File paths and constants ===
base_dir = r"C:\Users\japje\Documents\B.G\Project"
faiss_path = os.path.join(base_dir, "faiss_index_PaddleOCR.index")
csv_path = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2/chunked_text_PaddleOCR.csv")
text_folder = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2")
logo_path = os.path.join(base_dir, "Logo.png")
CACHE_FILE = "cache.json"
FEEDBACK_FILE = "feedback_log.csv"

# Together API config
TOGETHER_API_KEY = "tgp_v1_NOb5EFUR9QWlBnNwzFDjurHOokGKk3FsRtAKCmgr-sU"
TOGETHER_API_URL = "https://api.together.xyz/v1/chat/completions"
TOGETHER_HEADERS = {
    "Authorization": f"Bearer {TOGETHER_API_KEY}",
    "Content-Type": "application/json"
}

# Reward model path
reward_model_path = os.path.join(base_dir, r"C:\Users\japje\Downloads\japj\jap\reward_model\reward_model")

# === Load models ===
index = faiss.read_index(faiss_path)
df_chunks = pd.read_csv(csv_path)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
ocr_model = PaddleOCR(use_textline_orientation=True, lang='en')

# Load reward model
reward_tokenizer = AutoTokenizer.from_pretrained(reward_model_path)
reward_model = AutoModelForSequenceClassification.from_pretrained(reward_model_path)

# Chat log & cache
chat_log = []

# --- Cache helpers ---
def load_cache():
    return json.load(open(CACHE_FILE, "r", encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

def save_cache(cache):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4, ensure_ascii=False)

# --- Reward model scoring ---
def score_answer_with_reward_model(query, answer):
    input_text = query + " [SEP] " + answer
    inputs = reward_tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    with torch.no_grad():
        logits = reward_model(**inputs).logits
        prob = torch.softmax(logits, dim=1)[0][1].item()
    return prob

# --- Prompt builder ---
def build_prompt(chunks, question, chat_log=None):
    context = "\n".join(chunks["chunk_text"].tolist())
    history_text = ""
    if chat_log:
        for i, entry in enumerate(chat_log[-2:], start=1):
            history_text += f"Q{i}: {entry['question']}\nA{i}: {entry['answer']}\n"
    return (
        "You are a helpful assistant. Use the context below to answer.\n\n"
        f"{history_text}\n### Context ###\n{context}\n\n### Question ###\n{question}\n\n### Answer ###\n"
    )

# --- Together API call ---
def call_together_api(prompt):
    payload = {
        "model": "mistralai/Mistral-7B-Instruct-v0.1",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
        "max_tokens": 200
    }
    try:
        resp = requests.post(TOGETHER_API_URL, headers=TOGETHER_HEADERS, json=payload, timeout=20)
        if resp.status_code == 200:
            print("✅ Used Together API")
            answer = resp.json()["choices"][0]["message"]["content"].strip()
            return answer
        else:
            print(f"[API Error] {resp.status_code}: {resp.text}")
            return None
    except Exception as e:
        print(f"[API Exception] {e}")
        return None

# --- OCR ---
def extract_text_from_image(image_file):
    from PIL import Image
    img = Image.open(image_file)
    img_np = np.array(img)
    ocr_result = ocr_model.ocr(img_np)
    raw_text = "\n".join([line[1][0] for block in ocr_result for line in block]) if ocr_result else ""
    return raw_text

# --- Clean OCR text ---
def clean_ocr_text(ocr_text):
    lines = [line.strip() for line in ocr_text.split("\n") if len(line.strip()) > 2]
    cleaned = " ".join(lines)
    return cleaned if len(cleaned.split()) > 5 else None

# --- Search ---
def search_top_k(query, k=2, extra_text=None, image_file=None):
    embedding = embedding_model.encode([query])
    _, indices = index.search(np.array(embedding).astype("float32"), k)
    chunks = df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]].copy()
    if extra_text:
        try:
            uploaded_basename = os.path.basename(image_file.name) if image_file else "uploaded_image"
            uploaded_name = os.path.splitext(uploaded_basename)[0] + ".txt"
        except Exception:
            uploaded_name = "uploaded_image.txt"
        chunks = pd.concat([
            chunks,
            pd.DataFrame([{
                "chunk_id": "ocr_chunk",
                "filename": uploaded_name,
                "chunk_text": extra_text
            }])
        ], ignore_index=True)
    return chunks

# --- Best snippet ---
def find_best_snippet(chunks_df, question, max_length=250):
    question_vec = embedding_model.encode([question])[0]
    best_snip, best_file, best_score = "", "", -1
    for _, row in chunks_df.iterrows():
        sentences = re.split(r'(?<=[.!?]) +', row["chunk_text"])
        for sent in sentences:
            score = cosine_similarity([question_vec], [embedding_model.encode([sent])[0]])[0][0]
            if score > best_score:
                best_score, best_snip, best_file = score, sent, row["filename"]
    return best_file, best_snip[:max_length] + "..." if len(best_snip) > max_length else best_snip

# --- Generate multiple answers ---
def generate_candidate_answers(chunks, question, chat_log, num_candidates=3):
    candidates = []
    prompt = build_prompt(chunks, question, chat_log)
    for _ in range(num_candidates):
        answer = call_together_api(prompt)
        if answer:
            candidates.append(answer)
    _, snippet = find_best_snippet(chunks, question)
    fallback_answer = f"Based on the documents, here is some info:\n{snippet}"
    candidates.append(fallback_answer)
    return candidates

# --- Choose best answer ---
def choose_best_answer(query, candidates):
    scored = [(ans, score_answer_with_reward_model(query, ans)) for ans in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    print(f"Reward scores: {[round(s[1], 3) for s in scored]}")
    return scored[0][0]

# --- Chatbot main ---
def chatbot_ui(query, history, image_file):
    import hashlib
    cache = load_cache()
    ocr_raw = extract_text_from_image(image_file) if image_file else None
    ocr_text = clean_ocr_text(ocr_raw) if ocr_raw else None
    ocr_hash = hashlib.md5((ocr_text or '').encode('utf-8')).hexdigest()[:8] if ocr_text else "noocr"
    cache_key = f"{query}_{ocr_hash}"
    if cache_key in cache:
        answer = cache[cache_key]["answer"]
        source = cache[cache_key]["source"]
        snippet = cache[cache_key]["snippet"]
    else:
        chunks = search_top_k(query, k=2, extra_text=ocr_text, image_file=image_file)
        candidates = generate_candidate_answers(chunks, query, chat_log, num_candidates=3)
        answer = choose_best_answer(query, candidates)
        source, snippet = find_best_snippet(chunks, query)
        cache[cache_key] = {"answer": answer, "source": source, "snippet": snippet}
        save_cache(cache)
    chat_log.append({"question": query, "answer": answer, "source": source, "snippet": snippet})
    history.append((query, f"💬 Answer: {answer}\n📄 Source: {source}\n🔍 Snippet: {snippet}"))
    local_file = os.path.join(text_folder, source)
    os.makedirs("temp_docs", exist_ok=True)
    filepath = os.path.join("temp_docs", source)
    if os.path.exists(local_file):
        shutil.copy(local_file, filepath)
    else:
        filepath = None
    from gradio import update
    file_out = filepath if filepath and os.path.exists(filepath) else update(visible=False)
    return history, query, answer, source, file_out

# --- Export functions ---
def export_chat_to_csv():
    if not chat_log:
        return gr.update(value=None, visible=False)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = f"chat_export_{ts}.csv"
    keys = ["question", "answer", "source", "snippet"]
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(chat_log)
    return gr.update(value=csv_path, visible=True)

def export_feedback_to_csv():
    if not os.path.exists(FEEDBACK_FILE):
        return gr.update(value=None, visible=False)
    return gr.update(value=FEEDBACK_FILE, visible=True)

def export_chat_to_pdf():
    if not chat_log:
        return gr.update(value=None, visible=False)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_path = f"chat_export_{ts}.pdf"
    c = canvas.Canvas(pdf_path, pagesize=A4)
    width, height = A4
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, height - 60, "Chatter – Chat Summary Report")
    c.setFont("Helvetica", 10)
    c.drawString(50, height - 75, f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    if os.path.exists(logo_path):
        try:
            c.drawImage(ImageReader(logo_path), width - 100, height - 100, width=40, preserveAspectRatio=True, mask='auto')
        except Exception:
            pass
    y = height - 130
    c.setFont("Helvetica", 11)
    for i, entry in enumerate(chat_log, start=1):
        block = [
            f"{i}. Question: {entry['question']}",
            f"   Answer: {entry['answer']}",
            f"   Source: {entry['source']}",
            f"   Snippet: {entry['snippet']}",
        ]
        for line in block:
            for wrap_line in wrap(line, width=100):
                if y < 80:
                    c.showPage()
                    c.setFont("Helvetica", 11)
                    y = height - 80
                c.drawString(50, y, wrap_line)
                y -= 16
            y -= 6
    c.save()
    return gr.update(value=pdf_path, visible=True)

# --- Feedback functions ---
def load_feedback_summary():
    if not os.path.exists(FEEDBACK_FILE):
        return pd.DataFrame(columns=["query", "answer", "source", "feedback"])
    return pd.read_csv(FEEDBACK_FILE)

def save_feedback(query, answer, source, feedback):
    data = {"query": query, "answer": answer, "source": source, "feedback": feedback}
    file_exists = os.path.isfile(FEEDBACK_FILE)
    with open(FEEDBACK_FILE, "a", encoding="utf-8", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(data)

def feedback_fn(query, answer, source, feedback, history):
    save_feedback(query, answer, source, feedback)
    return history + [("✅ Feedback received: "+feedback.upper(), "")]

def show_feedback_summary():
    df = load_feedback_summary()
    return gr.update(value=df, visible=True)

# --- Reset ---
def reset_all():
    open(CACHE_FILE, "w").write("{}")
    open(FEEDBACK_FILE, "w").write("query,answer,source,feedback\n")
    shutil.rmtree("temp_docs", ignore_errors=True)
    chat_log.clear()
    return [], None, None, None, None, None, None, None, None

# --- UI ---
with gr.Blocks(title="Chatter with Reward Model + Fallback + Export & Feedback") as demo:
    gr.Markdown("## 🤖 Chatter - RAG + RLHF + Export & Feedback")
    chatbot = gr.Chatbot()
    query = gr.Textbox(label="Ask your question...")
    image_input = gr.File(label="Upload Image (for OCR)", file_types=["image"])
    file_viewer = gr.File(label="📄 View Document", visible=True)
    with gr.Row():
        btn_submit = gr.Button("Submit")
        btn_reset = gr.Button("Reset")
    with gr.Row():
        btn_good = gr.Button("👍")
        btn_bad = gr.Button("👎")
    with gr.Row():
        export_csv_btn = gr.Button("💾 Export Chat to CSV")
        export_file = gr.File(label="📥 Download Chat CSV", visible=False)
    with gr.Row():
        export_pdf_btn = gr.Button("🧾 Export Chat to PDF")
        pdf_file = gr.File(label="📄 Download Chat PDF", visible=False)
    with gr.Row():
        export_feedback_btn = gr.Button("📊 Export Feedback Log")
        export_feedback_file = gr.File(label="📥 Download Feedback CSV", visible=False)
    with gr.Row():
        feedback_summary_btn = gr.Button("📈 Show Feedback Summary")
        feedback_table = gr.DataFrame(headers=["query", "answer", "source", "feedback"], interactive=False, visible=False)
    state_query = gr.State()
    state_answer = gr.State()
    state_source = gr.State()
    btn_submit.click(chatbot_ui, inputs=[query, chatbot, image_input], outputs=[chatbot, state_query, state_answer, state_source, file_viewer])
    btn_reset.click(reset_all, inputs=[], outputs=[chatbot, file_viewer, state_query, state_answer, state_source, export_file, export_feedback_file, feedback_table, pdf_file])
    btn_good.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="good"), chatbot], outputs=[chatbot])
    btn_bad.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="bad"), chatbot], outputs=[chatbot])
    export_csv_btn.click(export_chat_to_csv, inputs=[], outputs=[export_file])
    export_pdf_btn.click(export_chat_to_pdf, inputs=[], outputs=[pdf_file])
    export_feedback_btn.click(export_feedback_to_csv, inputs=[], outputs=[export_feedback_file])
    feedback_summary_btn.click(show_feedback_summary, inputs=[], outputs=[feedback_table])

if __name__ == "__main__":
    demo.launch(share=True)


C:\Users\japje\anaconda3\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in C:\Users\japje\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None)
The model(UVDoc) is not supported to run in MKLDNN mode! Using `paddle` instead!
Using official model (UVDoc), the model files will be automatically downloaded and saved in C:\Users\japje\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in C:\Users\japje\.paddlex\official_models.


...

C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3

✅ Used Together API
✅ Used Together API
[API Error] 429: {
  "id": "o6hPRhD-4YNCb4-96d625963c6eabd9",
  "error": {
    "message": "Request was rejected due to request rate limiting. Your rate limits are 60 RPM (1 QPS) and 60000 TPM (1000 TPS). See details: https://docs.together.ai/docs/rate-limits",
    "type": "rate_limit",
    "param": null,
    "code": null
  }
}


C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3

Reward scores: [0.713, 0.688, 0.301]


C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\japje\anaconda3